# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/row-huh/ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Primary use:** Weekly priority queue for content ops / SEO team. Top 50 pages to review for title/snippet rewrite.

**Reason codes:**
- `high_score_long_article` — Model confidence ≥0.80, word_count ≥4000, scroll_rate <5%. Likely title/snippet misalignment or weak value prop.
- `high_score_strong_position` — Model confidence ≥0.80, position ≤10. Ranking is working; clicks are not. Metadata issue most likely.
- `moderate_score_stale` — Model confidence 0.5–0.8, content_age_days >300. Page may need refresh beyond metadata.
- `high_score_high_volume` — Model confidence ≥0.80, impressions ≥2000. High stakes; even small CTR lift compounds.
- `moderate_score_engagement_signal` — Model confidence 0.5–0.8, scroll_rate >10% (unusual for flagged page). Engagement is there; metadata may just need clarity.

**Output:** Ranked list of top 50 (or however many the team can handle per week), sorted by model score, with reason codes.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# Load and prep data (same as capstone)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
qualified = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].copy()
q = qualified[qualified.avg_position > 0].copy().reset_index(drop=True)

# Label and features
q["label"] = ((q.impressions_90d >= 500) & (q.avg_position <= 20) & (q.ctr < 0.5)).astype(int)
num_feats = ["avg_position", "impressions_90d", "word_count", "content_age_days",
             "days_since_last_update", "scroll_rate", "engagement_rate"]
cat_feats = ["content_type", "main_intent", "competition_level", "freshness_tier"]
q["has_word_count"] = q.word_count.notna().astype(int)

# Train/test split
X = q[num_feats + ["has_word_count"] + cat_feats].copy()
y = q["label"].values
gs = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gs.split(X, groups=q.client_id))

# Preprocessor with imputation
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), num_feats + ["has_word_count"]),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_feats)
])

# Train harder RF (without position/impressions, for honest signal)
harder_feats = [f for f in (num_feats + ["has_word_count"]) if f not in ["avg_position", "impressions_90d"]]
preprocessor_harder = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), harder_feats),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_feats)
])

X_train_harder = q.iloc[train_idx][harder_feats + cat_feats].copy()
X_test_harder = q.iloc[test_idx][harder_feats + cat_feats].copy()
y_train = y[train_idx]
y_test = y[test_idx]

rf_harder = Pipeline([
    ("prep", preprocessor_harder),
    ("clf", RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42))
])
rf_harder.fit(X_train_harder, y_train)
pred_harder = rf_harder.predict_proba(X_test_harder)[:, 1]

# Generate action playbook: top 50 with reason codes
q_test = q.iloc[test_idx].copy().reset_index(drop=True)
q_test["model_score"] = pred_harder

# Assign reason codes
def assign_reason_code(row):
    score = row["model_score"]
    wc = row["word_count"] if pd.notna(row["word_count"]) else 0
    sr = row["scroll_rate"] if pd.notna(row["scroll_rate"]) else 0
    pos = row["avg_position"]
    imp = row["impressions_90d"]
    age = row["content_age_days"]
    
    if score >= 0.80:
        if wc >= 4000 and sr < 5:
            return "high_score_long_article"
        elif pos <= 10:
            return "high_score_strong_position"
        elif imp >= 2000:
            return "high_score_high_volume"
        else:
            return "high_score"
    elif score >= 0.50:
        if age > 300:
            return "moderate_score_stale"
        elif sr > 10:
            return "moderate_score_engagement_signal"
        else:
            return "moderate_score"
    else:
        return "low_score"

q_test["reason_code"] = q_test.apply(assign_reason_code, axis=1)

# Top 50 ranked by score
top_50 = q_test.nlargest(50, "model_score")[[
    "content_id", "model_score", "reason_code", "avg_position", "ctr", 
    "impressions_90d", "word_count", "scroll_rate", "content_age_days"
]].reset_index(drop=True)

print("TOP 50 RANKED ACTIONS FOR REVIEW QUEUE")
print()
print(top_50.head(10).to_string())
print("...")
print()

print("REASON CODE BREAKDOWN (top 50):")
print(top_50["reason_code"].value_counts())
print()

print("EXPECTED TRUE POSITIVES (match label):")
print(f"  Pages matching label rule (impressions >= 500, position <= 20, CTR < 0.5%): {q_test.loc[top_50.index, 'label'].sum()} / 50")
print(f"  Precision@50: {q_test.loc[top_50.index, 'label'].mean():.3f}")

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended users:** Content ops / SEO team with weekly review capacity of ~50 pages.

**Intended use:** Prioritize a large backlog of pages for title/snippet review. Instead of working through all 9,759 low-CTR candidates in arbitrary order (195 weeks), focus on the ranked top-50 each week.

**Decision it supports:** "Which pages should we review this week for metadata clarity?"

**Decision it does NOT support:** 
- "Should we do a full content refresh?" — No, ranking alone doesn't tell you if content is low quality.
- "Will this fix CTR?" — No, no causal claim. Low CTR could be due to poor metadata, poor quality, or intent mismatch.
- "Pages to delete or deprioritize." — No, the ranking is about review, not suppression.

**Validity window:** ~90 days from the data snapshot. After that, rankings may drift (impressions shift, content ages, search behavior changes). Recommend retraining monthly or on significant traffic changes.

In [ ]:
# Validity check: how old is the data?
print("DATA FRESHNESS & VALIDITY:")
print()
print(f"Data snapshot is from: FlyRank starter export (90-day trailing window)")
print(f"Training window: fixed 90d @ export date")
print(f"Recommended re-eval: every 30 days or when...")
print(f"  - Top-50 recommendations yield <40% true positives (precision drops below threshold)")
print(f"  - Traffic volume shifts >20% month-over-month")
print(f"  - Search behavior changes (seasonal, algorithm update, market shift)")
print()
print("OUT-OF-SCOPE DECISIONS:")
print("  - Full content rewrites (use content audit, expert review, user intent analysis)")
print("  - Content deletion (use traffic, quality, & search intent analysis)")
print("  - Keyword strategy (use keyword research, trend analysis, SEM tools)")
print("  - Ranking predictions (model does not predict Google's algorithm)")

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before rewriting a title/snippet, reviewers MUST:**
1. Check the current title against the search intent. Does it clearly state what the page offers?
2. Confirm position and impressions in Google Search Console (not just the model's data).
3. Look at the SERP for that intent—what do competitors' titles look like?
4. Spot-check the page content—is it actually relevant to that search intent, or is low CTR due to intent mismatch (not metadata)?
5. Check for any recent changes (traffic, ranking) that might explain the low CTR other than metadata.

**NEVER automate:**
- Title/snippet rewrite without human review (always review the draft)
- Suppressing or deleting a page based on this ranking (use proper suppression workflows)
- Setting bid strategy or media spend based on this ranking (separate SEO and SEM decisions)
- Using this for ranking predictions (it only ranks *within a pattern*, not predicts overall rank)

**Red flags—skip the page if:**
- Content is brand new (<30 days old) and still indexing/ranking volatility
- Search intent for this query is seasonal (e.g., "holiday deals" in off-season)
- Page is a brand page, legal page, or other non-content-marketing piece (different rewrite rules)
- Title was recently rewritten (<14 days) and data may be catching up
- Low CTR is due to searcher intent mismatch, not metadata (ask: could a different title even fix it?)

In [ ]:
# Human review checklist
checklist = {
    "Check": [
        "GSC position & impressions (vs model data)",
        "Search intent clarity (current title match)",
        "SERP competitors' titles",
        "Page content relevance to search intent",
        "Recent traffic or ranking changes",
        "Content age (>30d)",
        "Is this a brand/legal/non-marketing page?",
        "Title last rewritten (>14d ago)",
        "Seasonal intent (is now the right time to review?)"
    ],
    "If False/No": [
        "Data may have changed; skip if large delta",
        "Metadata rewrite appropriate; proceed",
        "Use SERP data to inform title/snippet rewrite",
        "If mismatch: skip (rewrite won't fix CTR)",
        "If recent changes: wait a week; data may catch up",
        "Skip; ranking volatility too high",
        "Skip; different review process",
        "Wait 14 days; metric lag before rewrite",
        "Skip; wait for in-season opportunity"
    ]
}

checklist_df = pd.DataFrame(checklist)
print("HUMAN REVIEW CHECKLIST:")
print()
print(checklist_df.to_string(index=False))
print()
print("NO-GO LIST (Stop and skip the review if ANY of these are true):")
print("  1. Page <30 days old")
print("  2. Title was rewritten <14 days ago")
print("  3. Search intent is seasonal and we're outside the season")
print("  4. Page content doesn't actually match the search intent")
print("  5. Page is brand/legal/policy (not content marketing)")
print("  6. Position has changed >50% since data snapshot (dynamic ranking)")

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring metrics (track weekly):**
- **Precision@50 on new data:** Of the top 50 pages recommended this week, what % still fit the pattern (impressions ≥500, position ≤20, CTR <0.5%) when measured next week?
- **True positive feedback rate:** Of pages reviewers actually rewrote, what % showed improved CTR in the next 2 weeks? (Target: >40% improvement)
- **Traffic volume trends:** Month-over-month impression change across the dataset. If down >20%, rankings may shift and model needs retrain.
- **Reason code distribution:** If suddenly no "high_score_strong_position" pages and mostly "moderate_score" pages, the pattern may be shifting.

**Retrain triggers:**
1. **Precision drift:** Precision@50 on new data drops below 0.40 (from baseline 0.48).
2. **Outcome validation failure:** Reviewed pages don't show CTR improvement in follow-up window (sample 10 pages, see <30% improvement).
3. **Volume shift:** Total impressions across portfolio ±20% month-over-month.
4. **Calendar event:** Algorithm update, seasonal shift, or known market change.
5. **Time-based:** Monthly retrain on new 90-day window (if data pipeline allows).

**How to retrain:**
- Re-run w05_model.ipynb with new data
- Compare metrics to previous version
- If precision ≥0.45 on new test split, deploy
- If precision <0.45, investigate: label definition drift, feature importance shift, data quality change

In [ ]:
# Monitoring dashboard template
monitoring_template = pd.DataFrame({
    "Metric": [
        "Precision@50 (current week recommendations on next week's data)",
        "True positive feedback rate (pages improved CTR)",
        "Total portfolio impressions (MoM % change)",
        "Median CTR (high-score pages)",
        "Top reason code (% of top 50)",
        "Rerun time"
    ],
    "Baseline/Target": [
        "0.48",
        ">40% improved",
        "±5% normal, ±20% = alert",
        "0.07%",
        "high_score_strong_position ~30%",
        "Monthly or on trigger"
    ],
    "Alert (retrain) If": [
        "<0.40",
        "<25% improved",
        ">20% month-over-month",
        "Increases >50% (CTR inflation)",
        "Shifts >15% from baseline distribution",
        "30 days since last model run"
    ]
})

print("MONITORING DASHBOARD (Weekly check):")
print()
print(monitoring_template.to_string(index=False))
print()
print("RETRAIN DECISION TREE:")
print("  IF precision@50 < 0.40 on new data")
print("     → Retrain on latest 90-day window")
print("  ELSE IF true positive rate < 25%")
print("     → Investigate: are reviewers following the no-go list? Is pattern still real?")
print("  ELSE IF total impressions changed >20% MoM")
print("     → Major traffic shift: retrain recommended")
print("  ELSE")
print("     → Model is still good; keep using for this month")

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import os

# Ensure output directory
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

# Export 1: Top 50 ranked queue
top_50_export = top_50.copy()
top_50_export = top_50_export.round({"model_score": 3, "ctr": 4})
top_50_export.to_csv(f"{output_dir}/action_queue_top50.csv", index=False)
print(f"✓ Exported top-50 queue to {output_dir}/action_queue_top50.csv")

# Export 2: Reason code distribution
reason_dist = top_50["reason_code"].value_counts().to_frame("count")
reason_dist.to_csv(f"{output_dir}/reason_code_distribution.csv")
print(f"✓ Exported reason code distribution to {output_dir}/reason_code_distribution.csv")

# Export 3: Monitoring template (for weekly tracking)
monitoring_template.to_csv(f"{output_dir}/monitoring_template.csv", index=False)
print(f"✓ Exported monitoring template to {output_dir}/monitoring_template.csv")

# Export 4: Human review checklist
checklist_df.to_csv(f"{output_dir}/human_review_checklist.csv", index=False)
print(f"✓ Exported review checklist to {output_dir}/human_review_checklist.csv")

# Export 5: Summary stats for paper
summary_stats = pd.DataFrame({
    "Metric": [
        "Total pages in dataset",
        "Qualifying pages (impressions > 0, age >= 90d)",
        "Pages matching CTR opportunity pattern",
        "Weekly review capacity",
        "Weeks to clear backlog (unranked)",
        "Top 50 recommendations precision",
        "Model feature set size",
        "Train/test clients split"
    ],
    "Value": [
        f"{len(df):,}",
        f"{len(q):,}",
        f"{q['label'].sum():,}",
        "50",
        f"{round(q['label'].sum() / 50)}",
        f"{q_test.loc[top_50.index, 'label'].mean():.1%}",
        "12 (8 numeric + 4 categorical)",
        f"{len(set(q.iloc[train_idx].client_id))} / {len(set(q.iloc[test_idx].client_id))}"
    ]
})
summary_stats.to_csv(f"{output_dir}/capstone_summary_stats.csv", index=False)
print(f"✓ Exported summary stats to {output_dir}/capstone_summary_stats.csv")

print()
print("ALL EXPORTS READY FOR RESEARCH PAPER")
print(f"  Location: {output_dir}/")
print("  Files:")
print("    - action_queue_top50.csv (the ranked list)")
print("    - reason_code_distribution.csv (why each page is flagged)")
print("    - monitoring_template.csv (weekly metrics to track)")
print("    - human_review_checklist.csv (what reviewers must verify)")
print("    - capstone_summary_stats.csv (key numbers for the paper)")

---

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.